In [ ]:
import sys
import numpy as np
import spectral as sp
import matplotlib as mpl
import matplotlib.pyplot as plt

sys.path.append('../')
from preprocessing import *
from extinctions import *
from plotting_parameters import *
from skimage import filters

In [ ]:
bands_range = [520,900]

data_folder = "../../datasets/helicoid/012-01"
# data_folder = "../../datasets/helicoid/008-01"

class_labels = ["Not labled", "Normal", "Tumor", "Blood", "Background"]
class_colors = ["white", tum_blue_dark_2, tum_orange, tum_red, tum_grey_5]

# Load the sp data
img = sp.open_image(data_folder + "/raw.hdr")
white_ref = sp.open_image(data_folder + "/whiteReference.hdr")
dark_ref = sp.open_image(data_folder + "/darkReference.hdr")
gt_map = sp.open_image(data_folder + "/gtMap.hdr")

plot_img(img, gt_map, class_labels, class_colors, figsize=(5,5), legend=True)


img_interp, band_centers = bands_lin_interpolation(img, img.bands.centers, bands_range)
white_ref_interp, _ = bands_lin_interpolation(white_ref, img.bands.centers, bands_range)
dark_ref_interp, _ = bands_lin_interpolation(dark_ref, img.bands.centers, bands_range)
img_calib = calibrate_img(img_interp, white_ref_interp, dark_ref_interp)
img_calib_norm = img_calib / np.linalg.norm(img_calib, axis=(0,1), ord=1, keepdims=True)
img_smooth = smooth_spectral(img_calib_norm, 5)
absorbance = -np.log(img_smooth - np.min(img_smooth.flatten()) + 1e-8)

In [ ]:
bands_range = [520,900]
band_centers = np.arange(bands_range[0], bands_range[1]+1)
endmember_labels = ["oxCytc", "redCytc", "oxCytb", "redCytb", "oxCCO", "redCCO", "HHb", "HbO$_2$", "water", "lipids", "a'"]

extinction_dict = get_extinctions(bands_range)
cyt_c_ox = extinction_dict["cyt_c_ox"]
cyt_c_red = extinction_dict["cyt_c_red"]
cyt_b_ox = extinction_dict["cyt_b_ox"]
cyt_b_red = extinction_dict["cyt_b_red"]
cyt_oxi_ox = extinction_dict["cyt_oxi_ox"]
cyt_oxi_red = extinction_dict["cyt_oxi_red"]
hb = extinction_dict["hb"]
hbo2 = extinction_dict["hbo2"]
water = extinction_dict["water"]
fat = extinction_dict["fat"]
scatter_simple = (band_centers/500)**(-1.2)

spectra_folder = "../mc_sim/spectra_mc/"

hb_mc = np.loadtxt(spectra_folder + "m_hhb_50.txt")
hbo2_mc = np.loadtxt(spectra_folder + "m_hbo2_50.txt")

M_scatter_new = np.vstack((cyt_c_ox, cyt_c_red, cyt_b_ox, cyt_b_red, cyt_oxi_ox, cyt_oxi_red, hb_mc, hbo2_mc, water, fat, scatter_simple)).T
M_scatter_new_smooth = smooth_spectral(M_scatter_new.T,5).T

endmembers_remove = np.delete(M_scatter_new_smooth, 6, axis=1).T
data_hb_mc = osp(absorbance, endmembers_remove, hb_mc)

In [ ]:
plt.imshow(clip_data(data_hb_mc))

data_hb_mc_clip = clip_data(data_hb_mc)
data_hb_mc_clip_norm = (data_hb_mc_clip - np.min(data_hb_mc_clip.flatten())) / (np.max(data_hb_mc_clip.flatten()) - np.min(data_hb_mc_clip.flatten()))
data_hb_mc_clip_norm[data_hb_mc_clip_norm<0.5] = 0
data_hb_mc_clip_norm[data_hb_mc_clip_norm>0.5] = 1
plt.figure()
plt.imshow(data_hb_mc_clip_norm, cmap="gray")

In [ ]:
img_filtered = filters.frangi(clip_data(data_hb_mc), sigmas=(1,2,3), black_ridges=False)
print(img_filtered.shape)
plt.imshow(img_filtered)

img_filtered = filters.frangi(get_rgb(img), sigmas=(1,2,3), black_ridges=False)
print(img_filtered.shape)
plt.figure()
plt.imshow(img_filtered)

In [ ]:
img_filtered = filters.meijering(clip_data(data_hb_mc), black_ridges=False, sigmas=(1,2))
print(img_filtered.shape)
plt.imshow(img_filtered)

img_filtered[img_filtered<0.2] = 0
img_filtered[img_filtered>0.2] = 1
plt.figure()
plt.imshow(img_filtered, cmap="gray")

# overlay on the rgb image
img_rgb = get_rgb(img)
plt.figure()
plt.imshow(img_rgb)
plt.imshow(img_filtered, alpha=0.1, cmap="gray")

plt.figure()
plt.imshow(get_rgb(img))

In [ ]:
img_filtered = filters.sato(clip_data(data_hb_mc), black_ridges=False, sigmas=(1,2, 3, 4))
print(img_filtered.shape)
plt.imshow(img_filtered)

img_filtered[img_filtered<0.03] = 0
img_filtered[img_filtered>0.03] = 1
plt.figure()
plt.imshow(img_filtered, cmap="gray")

# overlay on the rgb image
img_rgb = get_rgb(img)
plt.figure()
plt.imshow(img_rgb)
plt.imshow(img_filtered, alpha=0.1, cmap="gray")

plt.figure()
plt.imshow(get_rgb(img))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

image_path = "012-02_fig5.png"
img = Image.open(image_path)
first_channel = img.split()[1]
img_array = np.array(first_channel)
img_array = img_array[40:400,520:900]

plt.figure()
plt.imshow(img_array, cmap="gray")

img_filtered_frangi = filters.frangi(img_array, sigmas=(1,2,3), black_ridges=True)
img_filtered_sato = filters.sato(img_array, sigmas=(1,2,3), black_ridges=True)
img_filtered_meijering = filters.meijering(img_array, black_ridges=True, sigmas=(1,2,3))


fig,ax = plt.subplots(1,3, figsize=(15,5))
ax[0].imshow(img_filtered_frangi)
ax[0].set_title("Frangi filter")
ax[1].imshow(img_filtered_sato)
ax[1].set_title("Sato filter")
ax[2].imshow(img_filtered_meijering)
ax[2].set_title("Meijering filter")


# normalize the images
img_filtered_frangi_norm = (img_filtered_frangi - np.min(img_filtered_frangi.flatten())) / (np.max(img_filtered_frangi.flatten()) - np.min(img_filtered_frangi.flatten()))
img_filtered_sato_norm = (img_filtered_sato - np.min(img_filtered_sato.flatten())) / (np.max(img_filtered_sato.flatten()) - np.min(img_filtered_sato.flatten()))
img_filtered_meijering_norm = (img_filtered_meijering - np.min(img_filtered_meijering.flatten())) / (np.max(img_filtered_meijering.flatten()) - np.min(img_filtered_meijering.flatten()))

threshold = 0.1
img_filtered_frangi_norm[img_filtered_frangi_norm<threshold] = 0
img_filtered_frangi_norm[img_filtered_frangi_norm>threshold] = 1
img_filtered_sato_norm[img_filtered_sato_norm<threshold] = 0
img_filtered_sato_norm[img_filtered_sato_norm>threshold] = 1
img_filtered_meijering_norm[img_filtered_meijering_norm<threshold] = 0
img_filtered_meijering_norm[img_filtered_meijering_norm>threshold] = 1

fig,ax = plt.subplots(1,3, figsize=(15,5))
ax[0].imshow(img_filtered_frangi_norm, cmap="gray")
ax[0].set_title("Frangi filter")
ax[1].imshow(img_filtered_sato_norm, cmap="gray")
ax[1].set_title("Sato filter")
ax[2].imshow(img_filtered_meijering_norm, cmap="gray")
ax[2].set_title("Meijering filter")


plt.figure()
plt.imshow(img_array)
# plt.imshow(img_filtered_meijering_norm, alpha=0.2)